In [1]:
!pip install pandasql
import pandas as pd
from pandasql import sqldf
import matplotlib.pyplot as plt

df = pd.read_csv('../data/customer_journey.csv')
pysql = lambda q: sqldf(q, globals())
print("Ready!")

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26799 sha256=797cd470b91126da140feed3207d8d5e2b1cc7016be79c50c9b6631533324751
  Stored in directory: c:\users\asus\appdata\local\pip\cache\wheels\b4\d0\8c\a6b366870bf041849cd96e03b71641e082f8d6456269b603b7
Successfully built pandasql


  DEPRECATION: Building 'pandasql' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pandasql'. Discussion can be found at https://github.com/pypa/pip/issues/6334


Ready!


In [2]:
q1 = """
SELECT 
    PageType,
    COUNT(DISTINCT UserID) AS unique_users
FROM df
WHERE PageType IN ('home','product_page','cart','checkout','confirmation')
GROUP BY PageType
"""
result1 = pysql(q1)
print(result1)

       PageType  unique_users
0          cart          1084
1      checkout           855
2  confirmation           792
3          home          1872
4  product_page          1763


In [3]:
q2 = """
SELECT
    PageType,
    COUNT(DISTINCT UserID) AS users,
    ROUND(COUNT(DISTINCT UserID) * 100.0 / 
        LAG(COUNT(DISTINCT UserID)) OVER (
            ORDER BY CASE PageType
                WHEN 'home'         THEN 1
                WHEN 'product_page' THEN 2
                WHEN 'cart'         THEN 3
                WHEN 'checkout'     THEN 4
                WHEN 'confirmation' THEN 5
            END), 2) AS conversion_from_prev
FROM df
GROUP BY PageType
"""
result2 = pysql(q2)
print(result2)

       PageType  users  conversion_from_prev
0          home   1872                   NaN
1  product_page   1763                 94.18
2          cart   1084                 61.49
3      checkout    855                 78.87
4  confirmation    792                 92.63


In [4]:
q3 = """
SELECT
    PageType,
    COUNT(DISTINCT UserID)                          AS users_reached,
    1872 - COUNT(DISTINCT UserID)                   AS users_lost,
    ROUND((1872 - COUNT(DISTINCT UserID)) * 100.0 
          / 1872, 2)                                AS pct_lost_from_start
FROM df
WHERE PageType IN ('home','product_page','cart','checkout','confirmation')
GROUP BY PageType
ORDER BY pct_lost_from_start
"""
result3 = pysql(q3)
print(result3)

       PageType  users_reached  users_lost  pct_lost_from_start
0          home           1872           0                 0.00
1  product_page           1763         109                 5.82
2          cart           1084         788                42.09
3      checkout            855        1017                54.33
4  confirmation            792        1080                57.69


In [5]:
q4 = """
SELECT
    DeviceType,
    COUNT(DISTINCT CASE WHEN PageType = 'home'         THEN UserID END) AS home_users,
    COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) AS converted_users,
    ROUND(COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN PageType = 'home'         THEN UserID END), 2) AS conversion_rate
FROM df
GROUP BY DeviceType
ORDER BY conversion_rate DESC
"""
result4 = pysql(q4)
print(result4)

  DeviceType  home_users  converted_users  conversion_rate
0     Mobile        1127              312            27.68
1     Tablet        1137              313            27.53
2    Desktop        1161              315            27.13


In [6]:
q5 = """
SELECT
    ReferralSource,
    COUNT(DISTINCT CASE WHEN PageType = 'home'         THEN UserID END) AS home_users,
    COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) AS converted_users,
    ROUND(COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN PageType = 'home'         THEN UserID END), 2) AS conversion_rate
FROM df
GROUP BY ReferralSource
ORDER BY conversion_rate DESC
"""
result5 = pysql(q5)
print(result5)

  ReferralSource  home_users  converted_users  conversion_rate
0         Google         969              262            27.04
1         Direct         925              230            24.86
2   Social Media         915              225            24.59
3          Email         940              230            24.47


In [7]:
q6 = """
SELECT
    Country,
    COUNT(DISTINCT CASE WHEN PageType = 'home'         THEN UserID END) AS total_users,
    COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) AS converted_users,
    ROUND(COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN PageType = 'home'         THEN UserID END), 2) AS conversion_rate
FROM df
GROUP BY Country
ORDER BY conversion_rate DESC
"""
result6 = pysql(q6)
print(result6)

     Country  total_users  converted_users  conversion_rate
0     France          615              165            26.83
1        USA          583              143            24.53
2      India          579              139            24.01
3         UK          630              141            22.38
4     Canada          600              134            22.33
5  Australia          571              127            22.24
6    Germany          602              124            20.60


In [10]:
queries = """-- Query 1: Funnel user count
SELECT 
    PageType,
    COUNT(DISTINCT UserID) AS unique_users
FROM df
WHERE PageType IN ('home','product_page','cart','checkout','confirmation')
GROUP BY PageType;

-- Query 2: Drop-off analysis
SELECT
    PageType,
    COUNT(DISTINCT UserID) AS users_reached,
    1872 - COUNT(DISTINCT UserID) AS users_lost,
    ROUND((1872 - COUNT(DISTINCT UserID)) * 100.0 / 1872, 2) AS pct_lost_from_start
FROM df
WHERE PageType IN ('home','product_page','cart','checkout','confirmation')
GROUP BY PageType
ORDER BY pct_lost_from_start;

-- Query 3: Conversion by device type
SELECT
    DeviceType,
    COUNT(DISTINCT CASE WHEN PageType = 'home' THEN UserID END) AS home_users,
    COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) AS converted_users,
    ROUND(COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN PageType = 'home' THEN UserID END), 2) AS conversion_rate
FROM df
GROUP BY DeviceType
ORDER BY conversion_rate DESC;

-- Query 4: Conversion by referral source
SELECT
    ReferralSource,
    COUNT(DISTINCT CASE WHEN PageType = 'home' THEN UserID END) AS home_users,
    COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) AS converted_users,
    ROUND(COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN PageType = 'home' THEN UserID END), 2) AS conversion_rate
FROM df
GROUP BY ReferralSource
ORDER BY conversion_rate DESC;

-- Query 5: Conversion by country
SELECT
    Country,
    COUNT(DISTINCT CASE WHEN PageType = 'home' THEN UserID END) AS total_users,
    COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) AS converted_users,
    ROUND(COUNT(DISTINCT CASE WHEN PageType = 'confirmation' THEN UserID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN PageType = 'home' THEN UserID END), 2) AS conversion_rate
FROM df
GROUP BY Country
ORDER BY conversion_rate DESC;
"""

import os
os.makedirs('../sql', exist_ok=True)

with open('../sql/funnel_queries.sql', 'w') as f:
    f.write(queries)

print("funnel_queries.sql saved at:", os.path.abspath('../sql/funnel_queries.sql'))

funnel_queries.sql saved at: C:\Users\Asus\ funnel_analysis\sql\funnel_queries.sql
